# ChuckleNet Phase A + Prosody: Streaming Audio (No RAM Cache)

## Key change from original:
- **STREAMS audio on-demand** instead of pre-caching all audio into RAM
- Uses `librosa.load(audio_path, offset=start, duration=5.0)` to read only the needed segment
- Eliminates OOM crashes on Colab T4 (~12GB RAM limit)

## Architecture (unchanged)
- **WavLM encoder** (FROZEN)
- **Attention pooling**
- **21-dim prosody** (pre-extracted)
- **Simple late fusion** → MLP classifier

## Expected: Val F1 ~0.75+ (same as original, but stable)

In [ ]:
# ============================================================
# SETUP
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/chuckle_checkpoints'
AUDIO_DIR = f'{BASE}/audio'
PROSODY_PATH = f'{BASE}/prosody_phaseD.json'
OUTPUT_DIR = f'{BASE}'
AUDIO_CKPT = f'{BASE}/phaseA_prosody_best.pt'

import os, json, time, numpy as np, librosa, torch
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import defaultdict
import gc

SR = 16000
MAX_DURATION = 5.0  # seconds to load per segment
MAX_LEN = int(MAX_DURATION * SR)  # 80000 samples
BATCH = 16  # Reduced from 32 to save memory
EPOCHS = 10
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('⚠️  WARNING: No GPU! Training will be very slow.')


In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================
HELD_OUT = {'BFIHCzw3itk', 'BAD4askmGgk', '1Nb3_os4RSA'}

print('Loading aligned_utterances...')
au_path = f'{BASE}/aligned_utterances.jsonl'
all_data = [json.loads(line) for line in open(au_path)]
print(f'  Total: {len(all_data)}')

held_out = [d for d in all_data if d['video_id'] in HELD_OUT]
in_dist = [d for d in all_data if d['video_id'] not in HELD_OUT]
print(f'  In-dist: {len(in_dist)}, Held-out test: {len(held_out)}')

np.random.seed(SEED)
np.random.shuffle(in_dist)
n_train = int(len(in_dist) * 0.85)
train, val = in_dist[:n_train], in_dist[n_train:]
test = held_out

print(f'  Train: {len(train)}, Val: {len(val)}, Test: {len(test)}')
print(f'  Train pos%={100*sum(d["label_any"] for d in train)/len(train):.1f}%')
print(f'  Val pos%={100*sum(d["label_any"] for d in val)/len(val):.1f}%')
print(f'  Test pos%={100*sum(d["label_any"] for d in test)/len(test):.1f}%')

assert len(set(d['video_id'] for d in train) & set(d['video_id'] for d in held_out)) == 0, 'Leak!'
print('  ✅ No held-out leakage')


In [ ]:
# ============================================================
# LOAD PROSODY CACHE (small, ~5MB)
# ============================================================
print('Loading prosody cache...')
with open(PROSODY_PATH) as f:
    prosody_raw = json.load(f)
prosody_map = {d['uid']: d['feats'] for d in prosody_raw}
print(f'  {len(prosody_map)} prosody entries, dim={len(prosody_raw[0]["feats"])}')

from sklearn.preprocessing import StandardScaler
train_prosody = np.array([prosody_map.get(f"{d['video_id']}_{d['start']:.2f}", np.zeros(21)) for d in train])
prosody_scaler = StandardScaler()
prosody_scaler.fit(train_prosody)
print(f'  Prosody scaler fitted on {len(train_prosody)} samples')

def get_prosody(d):
    key = f"{d['video_id']}_{d['start']:.2f}"
    feats = prosody_map.get(key, np.zeros(21))
    return prosody_scaler.transform([feats])[0]


In [ ]:
# ============================================================
# STREAMING AUDIO LOADER (no RAM cache!)
# ============================================================

def load_audio_segment(d, sr=SR, max_duration=MAX_DURATION):
    """
    Load only the needed audio segment on-demand.
    Much more RAM-efficient than pre-caching all audio.
    """
    vid = d['video_id']
    start = d['start']
    audio_path = f'{AUDIO_DIR}/{vid}.mp3'
    
    try:
        # Stream only the segment we need (offset + duration)
        # Add small offset to capture pre-laugh audio context
        offset = max(0, start - 0.05)
        audio, _ = librosa.load(
            audio_path,
            sr=sr,
            mono=True,
            offset=offset,
            duration=max_duration
        )
        
        # Pad if audio is shorter than expected
        target_len = int(max_duration * sr)
        if len(audio) < target_len:
            audio = np.pad(audio, (0, target_len - len(audio)))
        
        return audio[:target_len]  # Ensure exact length
        
    except Exception as e:
        print(f'  Warning: could not load {vid} at {start:.2f}s: {e}')
        return np.zeros(target_len, dtype=np.float32)

# Test the streaming loader
print('Testing streaming audio loader...')
test_audio = load_audio_segment(train[0])
print(f'  Loaded segment shape: {test_audio.shape}, dtype: {test_audio.dtype}')
print('  ✅ Streaming audio loader ready')


In [ ]:
# ============================================================
# LOAD WavLM ENCODER (FROZEN)
# ============================================================
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

print('Loading WavLM (FROZEN)...')
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
wavlm.eval()
for p in wavlm.parameters():
    p.requires_grad = False
wavlm = wavlm.to(device)
print(f'  WavLM: frozen, {sum(p.numel() for p in wavlm.parameters()):,} params')

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')
print('  Feature extractor ready')


In [ ]:
# ============================================================
# ATTENTION POOLING + MODEL
# ============================================================
import torch.nn as nn

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
        
    def forward(self, hidden_states):
        weights = self.attn(hidden_states).squeeze(-1)
        weights = torch.softmax(weights, dim=-1)
        pooled = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)
        return pooled, weights

class PhaseAWithProsody(nn.Module):
    def __init__(self, encoder, prosody_dim=21, hidden=128):
        super().__init__()
        self.encoder = encoder
        self.audio_pool = AttentionPooling(768)
        
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )
        
        total_dim = 768 + 32
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(total_dim, hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 64),
            nn.GELU(),
            nn.Linear(64, 2)
        )
        
    def forward(self, waveforms, prosody_feats):
        # Waveforms: (batch, 80000) - 5 seconds at 16kHz
        # Prosody: (batch, 21)
        
        # 1. WavLM encoder (frozen)
        wav_list = [w.cpu().numpy() for w in waveforms]
        inputs = feature_extractor(wav_list, sampling_rate=SR, return_tensors='pt', padding=False)
        inputs = {k: v.to(waveforms.device) for k, v in inputs.items()}
        with torch.no_grad():
            hidden = self.encoder(**inputs).last_hidden_state
        
        # 2. Attention pooling
        audio_emb, attn_weights = self.audio_pool(hidden)
        
        # 3. Prosody projection
        prosody_emb = self.prosody_proj(prosody_feats)
        
        # 4. Late fusion + classify
        fused = torch.cat([audio_emb, prosody_emb], dim=1)
        logits = self.classifier(fused)
        
        return logits

model = PhaseAWithProsody(wavlm, prosody_dim=21, hidden=128).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {n_params:,} total | {n_trainable:,} trainable')


In [ ]:
# ============================================================
# OPTIMIZER & LOSS
# ============================================================
classifier_params = (
    list(model.audio_pool.parameters()) +
    list(model.prosody_proj.parameters()) +
    list(model.classifier.parameters())
)

optimizer = torch.optim.AdamW([
    {'params': classifier_params, 'lr': 1e-3, 'weight_decay': 0.01}
], lr=1e-3, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Class weights [1, 2.5] as in original
class_weights = torch.tensor([1.0, 2.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print('Optimizer & scheduler ready')


In [ ]:
# ============================================================
# TRAINING LOOP (streaming audio)
# ============================================================
def train_epoch(model, data, optimizer, criterion, device, batch_size=BATCH):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    np.random.seed(SEED + epoch)
    indices = np.random.permutation(len(data))
    
    for i in range(0, len(data), batch_size):
        batch_idx = indices[i:i+batch_size]
        batch = [data[j] for j in batch_idx]
        
        # Load audio on-demand (streaming)
        waveforms = []
        prosody_feats = []
        labels = []
        
        for d in batch:
            audio = load_audio_segment(d)
            waveforms.append(torch.FloatTensor(audio).unsqueeze(0))
            prosody_feats.append(get_prosody(d))
            labels.append(d['label_any'])
        
        waveforms = torch.stack(waveforms).squeeze(-1).to(device)  # (batch, 80000)
        prosody_feats = torch.FloatTensor(np.array(prosody_feats)).to(device)  # (batch, 21)
        labels = torch.LongTensor(labels).to(device)  # (batch,)
        
        optimizer.zero_grad()
        logits = model(waveforms, prosody_feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
        # Clear memory
        del waveforms, prosody_feats, labels
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    
    avg_loss = total_loss / (len(data) / batch_size)
    f1 = f1_score(all_labels, all_preds, average='binary')
    return avg_loss, f1

def eval_epoch(model, data, criterion, device, batch_size=BATCH):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        
        waveforms = []
        prosody_feats = []
        labels = []
        
        for d in batch:
            audio = load_audio_segment(d)
            waveforms.append(torch.FloatTensor(audio).unsqueeze(0))
            prosody_feats.append(get_prosody(d))
            labels.append(d['label_any'])
        
        waveforms = torch.stack(waveforms).squeeze(-1).to(device)
        prosody_feats = torch.FloatTensor(np.array(prosody_feats)).to(device)
        labels = torch.LongTensor(labels).to(device)
        
        with torch.no_grad():
            logits = model(waveforms, prosody_feats)
            loss = criterion(logits, labels)
        
        total_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
        del waveforms, prosody_feats, labels
        gc.collect()
    
    avg_loss = total_loss / (len(data) / batch_size)
    f1 = f1_score(all_labels, all_preds, average='binary')
    return avg_loss, f1, all_preds, all_labels

print('Training functions defined')


In [ ]:
# ============================================================
# TRAIN!
# ============================================================
print('Starting training...')
print(f'  Train: {len(train)}, Val: {len(val)}, Test: {len(test)}')
print(f'  Batch: {BATCH}, Epochs: {EPOCHS}')
print()

best_val_f1 = 0
best_state = None
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}

t0 = time.time()
for epoch in range(EPOCHS):
    epoch_t0 = time.time()
    
    train_loss, train_f1 = train_epoch(model, train, optimizer, criterion, device)
    val_loss, val_f1, _, _ = eval_epoch(model, val, criterion, device)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    
    epoch_time = time.time() - epoch_t0
    total_time = time.time() - t0
    
    print(f'Epoch {epoch+1}/{EPOCHS} ({epoch_time:.0f}s, total {total_time/60:.1f}m)')
    print(f'  Train loss={train_loss:.4f} f1={train_f1:.4f}')
    print(f'  Val   loss={val_loss:.4f} f1={val_f1:.4f}')
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'  ✅ New best! Val F1={best_val_f1:.4f}')
    print()

print(f'Training done! Best val F1: {best_val_f1:.4f}')
print(f'Total time: {(time.time()-t0)/60:.1f} minutes')


In [ ]:
# ============================================================
# EVALUATE ON HELD-OUT TEST SET
# ============================================================
if best_state is not None:
    model.load_state_dict(best_state)

print('Evaluating on held-out test set...')
test_loss, test_f1, test_preds, test_labels = eval_epoch(model, test, criterion, device)

print(f'  Test loss={test_loss:.4f} F1={test_f1:.4f}')
print(f'  Precision={precision_score(test_labels, test_preds):.4f}')
print(f'  Recall={recall_score(test_labels, test_preds):.4f}')

# Save results
results = {
    'val_f1': best_val_f1,
    'test_f1': test_f1,
    'test_precision': precision_score(test_labels, test_preds),
    'test_recall': recall_score(test_labels, test_preds),
    'history': history
}

with open(f'{OUTPUT_DIR}/phaseA_prosody_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {OUTPUT_DIR}/phaseA_prosody_results.json')


In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================
if best_state is not None:
    torch.save(best_state, AUDIO_CKPT)
    print(f'Model saved to {AUDIO_CKPT}')
else:
    torch.save(model.state_dict(), AUDIO_CKPT)
    print(f'Model saved (no improvement found) to {AUDIO_CKPT}')
